In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from scipy.interpolate import interp1d

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from implied_vol import implied_vol

os.makedirs('../figures', exist_ok=True)
sns.set_theme(style='darkgrid')
print('Setup OK')


Setup OK


# Surface de Volatilité Implicite 3D — SPY

Ce notebook construit la **surface de volatilité implicite** $\sigma(K/S, T)$ à partir de
données de marché réelles (SPY via `yfinance`).

La surface est organisée en deux dimensions :
- **Moneyness** $K/S$ (axe des strikes normalisé) — révèle le **skew** à chaque maturité.
- **Maturité** $T$ en années — révèle la **term structure** de volatilité.

Un praticien y lit simultanément le skew d'indice (queues épaisses, prime de krach)
et la structure par terme (vol court terme plus élevée que vol long terme en période de stress).

## 1. Collecte des données — 6 maturités SPY

On sélectionne 6 expirations réparties de ~1 mois à ~12 mois.
Pour chaque maturité on récupère la chaîne complète (calls + puts).


In [2]:
import yfinance as yf
from datetime import datetime

TICKER = 'SPY'
tk     = yf.Ticker(TICKER)
S0     = tk.fast_info['lastPrice']
today  = datetime.today()

# Cibles : ~1m, ~2m, ~3m, ~5m, ~9m, ~12m
TARGET_EXPIRIES = [
    '2026-07-31',   # ~1 mois
    '2026-08-21',   # ~2 mois
    '2026-09-18',   # ~3 mois
    '2026-11-20',   # ~5 mois
    '2027-03-19',   # ~9 mois
    '2027-06-17',   # ~12 mois
]

R = 0.05  # taux sans risque approximatif

chains = {}
for exp in TARGET_EXPIRIES:
    T_exp = (datetime.strptime(exp, '%Y-%m-%d') - today).days / 365.0
    ch    = tk.option_chain(exp)
    c     = ch.calls.copy();  c['mid'] = (c['bid'] + c['ask']) / 2
    p     = ch.puts.copy();   p['mid'] = (p['bid'] + p['ask']) / 2
    chains[exp] = {'T': T_exp, 'calls': c, 'puts': p}
    print(f'{exp}  T={T_exp:.3f}a  calls={len(c)}  puts={len(p)}')

print(f'\nTicker : {TICKER}   Spot S0 = {S0:.2f}')
print(f'Maturités chargées : {len(chains)}')


2026-07-31  T=0.082a  calls=242  puts=227
2026-08-21  T=0.140a  calls=238  puts=209


2026-09-18  T=0.216a  calls=267  puts=285
2026-11-20  T=0.389a  calls=105  puts=108


2027-03-19  T=0.715a  calls=193  puts=129
2027-06-17  T=0.962a  calls=211  puts=171

Ticker : SPY   Spot S0 = 741.00
Maturités chargées : 6


## 2. Calcul des volatilités implicites par strike

**Choix call / put** : on utilise l'option *hors de la monnaie* pour chaque strike,
car les options OTM sont plus liquides (bid–ask resserré) et donnent une meilleure qualité d'IV.

- Puts OTM pour $K/S < 1$
- Calls OTM pour $K/S \geq 1$

**Filtres appliqués :**
1. Moneyness $\in [0.80, 1.20]$
2. Mid-prix $> 0$ et bid $> 0$
3. Spread $\leq 30\%$ du mid
4. IV $\in [1\%, 150\%]$ et non NaN

In [3]:
MONO_LO, MONO_HI = 0.80, 1.20

iv_data = {}  # exp -> DataFrame(moneyness, IV)

for exp, d in chains.items():
    T_exp = d['T']
    rows  = []

    for df_opt, opt_type in [(d['puts'], 'put'), (d['calls'], 'call')]:
        for _, row in df_opt.iterrows():
            K   = float(row['strike'])
            m   = K / S0
            mid = float(row.get('mid', np.nan))
            bid = float(row.get('bid', np.nan))

            if not (MONO_LO <= m <= MONO_HI):
                continue
            # Sélection OTM
            if opt_type == 'put'  and m >= 1.0:
                continue
            if opt_type == 'call' and m <  1.0:
                continue

            if np.isnan(mid) or mid <= 0 or np.isnan(bid) or bid <= 0:
                continue
            spread = float(row.get('ask', mid)) - bid
            if spread > 0.30 * mid:
                continue

            iv = implied_vol(mid, S0, K, T_exp, R, opt_type)
            if np.isnan(iv) or iv < 0.01 or iv > 1.50:
                continue

            rows.append({'moneyness': m, 'IV': iv, 'strike': K, 'type': opt_type})

    df = pd.DataFrame(rows).sort_values('moneyness').drop_duplicates('moneyness')
    iv_data[exp] = df
    print(f'{exp}  T={T_exp:.3f}a  points valides : {len(df)}'
          f'  IV range [{df["IV"].min():.2%} – {df["IV"].max():.2%}]')


2026-07-31  T=0.082a  points valides : 194  IV range [12.19% – 37.00%]
2026-08-21  T=0.140a  points valides : 176  IV range [12.23% – 32.42%]


2026-09-18  T=0.216a  points valides : 238  IV range [12.30% – 30.32%]
2026-11-20  T=0.389a  points valides : 59  IV range [12.62% – 28.24%]
2027-03-19  T=0.715a  points valides : 59  IV range [12.98% – 26.87%]
2027-06-17  T=0.962a  points valides : 59  IV range [13.37% – 26.56%]


## 3. Construction de la grille commune

Pour former une surface régulière, on interpole chaque courbe IV sur une grille de moneyness
commune $[0.80, 1.20]$ à 40 points équidistants.
On utilise une interpolation linéaire par maturité, extrapolation plate aux extrêmes
(valeur du bord la plus proche — conservative, évite les artefacts).

In [4]:
MONO_GRID = np.linspace(MONO_LO, MONO_HI, 40)
maturities = list(chains.keys())
T_values   = np.array([chains[e]['T'] for e in maturities])

IV_matrix = np.full((len(maturities), len(MONO_GRID)), np.nan)

for i, exp in enumerate(maturities):
    df = iv_data[exp]
    if len(df) < 3:
        print(f'[skip] {exp}: seulement {len(df)} points')
        continue
    f = interp1d(df['moneyness'], df['IV'],
                 kind='linear', bounds_error=False,
                 fill_value=(df['IV'].iloc[0], df['IV'].iloc[-1]))
    IV_matrix[i] = f(MONO_GRID)

print(f'Grille moneyness : {len(MONO_GRID)} points  [{MONO_GRID[0]:.2f}, {MONO_GRID[-1]:.2f}]')
print(f'Maturités        : {len(T_values)} valeurs  T = {T_values.round(3)}')
print(f'Matrice IV       : {IV_matrix.shape}  (maturités × moneyness)')
print(f'IV min={np.nanmin(IV_matrix):.2%}  max={np.nanmax(IV_matrix):.2%}')


Grille moneyness : 40 points  [0.80, 1.20]
Maturités        : 6 valeurs  T = [0.082 0.14  0.216 0.389 0.715 0.962]
Matrice IV       : (6, 40)  (maturités × moneyness)
IV min=12.20%  max=37.00%


## 4. Surface de volatilité 3D (Plotly)

La surface $\sigma(K/S, T)$ visualise simultanément :
- Le **skew** (pente négative selon $K/S$) : puts OTM plus chers que calls OTM.
- La **term structure** (évolution selon $T$) : la vol court terme est souvent plus élevée
  que la vol long terme en régime de stress (structure inversée), ou plus basse en régime normal.
- L'**ATM vol curve** le long de la diagonale $K/S = 1$.


In [5]:
X, Y = np.meshgrid(MONO_GRID, T_values)
Z    = IV_matrix * 100  # en %

fig3d = go.Figure(data=[go.Surface(
    x=X, y=Y, z=Z,
    colorscale='RdYlGn_r',
    colorbar=dict(title='IV (%)', thickness=15),
    hovertemplate=(
        'Moneyness: %{x:.3f}<br>'
        'Maturité: %{y:.3f} a<br>'
        'IV: %{z:.1f}%<extra></extra>'
    ),
)])

fig3d.update_layout(
    title=dict(text=f'Surface de Volatilité Implicite — {TICKER}', font_size=16),
    scene=dict(
        xaxis=dict(title='Moneyness K/S'),
        yaxis=dict(title='Maturité (années)'),
        zaxis=dict(title='Vol implicite (%)'),
        camera=dict(eye=dict(x=1.6, y=-1.6, z=0.8)),
    ),
    width=900, height=650,
    margin=dict(l=20, r=20, t=60, b=20),
)

html_path = '../figures/08_vol_surface.html'
png_path  = '../figures/08_vol_surface.png'

fig3d.write_html(html_path)
fig3d.write_image(png_path, width=1200, height=750, scale=2)

fig3d.show()
print(f'HTML : figures/08_vol_surface.html')
print(f'PNG  : figures/08_vol_surface.png')


HTML : figures/08_vol_surface.html
PNG  : figures/08_vol_surface.png


## 5. Comparaison 2D — skew par maturité

On superpose les courbes IV pour trois maturités (courte / moyenne / longue) afin de montrer
l'**aplatissement du skew avec la maturité** : le skew est plus prononcé sur les options
court terme (plus grande sensibilité aux queues de distribution sur un horizon court) et
tend à s'atténuer pour les maturités longues (le temps lisse les chocs extrêmes).


In [6]:
# Sélection : courte (0), moyenne (2), longue (5)
idx_sel  = [0, 2, 5]
labels   = [f'{maturities[i]}  (T={T_values[i]:.2f}a)' for i in idx_sel]
colors   = ['#e74c3c', '#2980b9', '#27ae60']

fig2, ax = plt.subplots(figsize=(10, 5))

for i, (idx, label, col) in enumerate(zip(idx_sel, labels, colors)):
    iv_row = IV_matrix[idx] * 100
    ax.plot(MONO_GRID, iv_row, '-o', ms=3, lw=2, color=col, label=label)

ax.axvline(1.0, color='grey', lw=1, ls=':', alpha=0.7)
ax.set_title(f'Skew de volatilité implicite par maturité — {TICKER}', fontsize=13)
ax.set_xlabel('Moneyness  K / S')
ax.set_ylabel('Volatilité implicite (%)')
ax.legend(title='Maturité')

# Annotation pente skew sur la maturité courte
m_lo  = MONO_GRID[5];   iv_lo  = IV_matrix[idx_sel[0]][5]  * 100
m_hi  = MONO_GRID[-5];  iv_hi  = IV_matrix[idx_sel[0]][-5] * 100
slope = (iv_hi - iv_lo) / (m_hi - m_lo)
ax.annotate(f'Pente ≈ {slope:.0f} vol pts / Δmoneyness',
            xy=(MONO_GRID[10], IV_matrix[idx_sel[0]][10] * 100),
            xytext=(0.82, IV_matrix[idx_sel[0]][-8] * 100 * 0.90),
            fontsize=9, color='#e74c3c',
            arrowprops=dict(arrowstyle='->', color='#e74c3c'))

plt.tight_layout()
png2_path = '../figures/09_skew_by_maturity.png'
plt.savefig(png2_path, dpi=150)
plt.show()
print(f'PNG : figures/09_skew_by_maturity.png')


PNG : figures/09_skew_by_maturity.png


C:\Users\octav\AppData\Local\Temp\ipykernel_28104\3459446104.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Term structure ATM


In [7]:
# Indice du strike le plus proche de ATM (moneyness = 1.0)
atm_idx = np.argmin(np.abs(MONO_GRID - 1.0))
iv_atm_term = IV_matrix[:, atm_idx] * 100  # % pour chaque maturité

fig_ts, ax_ts = plt.subplots(figsize=(8, 4))
ax_ts.plot(T_values, iv_atm_term, 'o-', color='steelblue', lw=2, ms=7)
for T_i, iv_i in zip(T_values, iv_atm_term):
    ax_ts.annotate(f'{iv_i:.1f}%', (T_i, iv_i),
                   textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)

ax_ts.set_title(f'Term Structure de la vol ATM — {TICKER}', fontsize=13)
ax_ts.set_xlabel('Maturité (années)')
ax_ts.set_ylabel('Vol implicite ATM (%)')

plt.tight_layout()
png_ts = '../figures/10_atm_term_structure.png'
plt.savefig(png_ts, dpi=150)
plt.show()
print(f'PNG : figures/10_atm_term_structure.png')
print()
print('Term structure ATM :')
for exp, T_i, iv_i in zip(maturities, T_values, iv_atm_term):
    print(f'  {exp}  T={T_i:.3f}a  IV_ATM={iv_i:.2f}%')


PNG : figures/10_atm_term_structure.png

Term structure ATM :
  2026-07-31  T=0.082a  IV_ATM=16.08%
  2026-08-21  T=0.140a  IV_ATM=16.16%
  2026-09-18  T=0.216a  IV_ATM=16.85%
  2026-11-20  T=0.389a  IV_ATM=18.15%
  2027-03-19  T=0.715a  IV_ATM=19.71%
  2027-06-17  T=0.962a  IV_ATM=20.71%


C:\Users\octav\AppData\Local\Temp\ipykernel_28104\1467460870.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Ce que lit un trader sur la surface

| Dimension | Ce qu'on observe | Interprétation |
|-----------|-----------------|----------------|
| **Skew** ($K/S$) | IV décroissante de gauche (puts OTM) à droite (calls OTM) | Prime de risque de krach : protection put coûteuse, log-normalité invalide |
| **Term structure** ($T$) | Vol court terme > vol long terme (structure inversée) | Stress court terme ou événement attendu ; la vol long terme est ancrée par la moyenne historique |
| **Aplatissement** | Le skew s'atténue avec $T$ | Sur un horizon long, les queues de distribution s'homogénéisent — le TCL «lisse» les chocs |
| **Niveau ATM** | La diagonale $K/S=1$ | Niveau de vol «de marché» ; comparer à la vol réalisée pour décider d'acheter ou vendre de la volatilité |

### Pourquoi la surface n'est pas plane ?

Black-Scholes suppose **une seule volatilité constante** $\sigma$. Une surface plate signifierait
que le marché croit en la log-normalité pour tous les strikes et toutes les maturités.
En réalité, la surface non plate encode :
1. La **distribution réelle** des rendements : asymétrie négative, queues épaisses (leptokurtosis).
2. Les **anticipations de volatilité future** : la term structure reflète la variance forward
   implicite entre deux maturités.
3. L'**offre et la demande** : les fonds de couverture achètent massivement des puts OTM
   (protection de portefeuille), gonflant leur prime.

La surface est l'objet central du *vol trading* : on l'arbitrage, on la modèle (Heston, SABR,
SVI) et on s'en sert pour hedger les books d'exotiques.


In [8]:
print('=== Résumé de la surface ===')
print(f'Sous-jacent       : {TICKER}   Spot S0 = {S0:.2f}')
print(f'Maturités         : {len(maturities)}  ({T_values[0]:.3f}a → {T_values[-1]:.3f}a)')
print(f'Grille moneyness  : {len(MONO_GRID)} points  [{MONO_GRID[0]:.2f}, {MONO_GRID[-1]:.2f}]')
print(f'Matrice IV        : {IV_matrix.shape}')
print(f'IV globale min    : {np.nanmin(IV_matrix)*100:.2f}%')
print(f'IV globale max    : {np.nanmax(IV_matrix)*100:.2f}%')
print()
print('Fichiers produits :')
for f in ['08_vol_surface.html', '08_vol_surface.png',
          '09_skew_by_maturity.png', '10_atm_term_structure.png']:
    path = f'../figures/{f}'
    exists = os.path.exists(path)
    size   = os.path.getsize(path) if exists else 0
    print(f'  figures/{f}  {"OK" if exists else "MANQUANT"}  ({size:,} bytes)')


=== Résumé de la surface ===
Sous-jacent       : SPY   Spot S0 = 741.00
Maturités         : 6  (0.082a → 0.962a)
Grille moneyness  : 40 points  [0.80, 1.20]
Matrice IV        : (6, 40)
IV globale min    : 12.20%
IV globale max    : 37.00%

Fichiers produits :
  figures/08_vol_surface.html  OK  (4,865,554 bytes)
  figures/08_vol_surface.png  OK  (506,516 bytes)
  figures/09_skew_by_maturity.png  OK  (115,188 bytes)
  figures/10_atm_term_structure.png  OK  (49,596 bytes)
